In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import gradio as gr
from PIL import Image
import timm
from torchvision import transforms

# 1. SETUP & LOAD MODEL
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/kaggle/input/notebooka20ecd0c3f/realistic_alz_model.pth"

# Re-initialize the same architecture used in training
model = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=4)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

class_names = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

# 2. GRAD-CAM CORE LOGIC
def generate_gradcam(input_tensor, target_class):
    # We target the last attention block of the ViT
    # For ViT-Tiny, this is usually model.blocks[-1].norm1
    feature_map = None
    gradients = None

    def save_gradient(grad):
        nonlocal gradients
        gradients = grad

    # Hook into the last block's normalization layer
    target_layer = model.blocks[-1].norm1
    
    # Forward pass hook
    def hook_fn(module, input, output):
        nonlocal feature_map
        feature_map = output

    hook = target_layer.register_forward_hook(hook_fn)
    
    # Forward pass
    output = model(input_tensor)
    output[:, target_class].backward()
    
    # Register backward hook to get gradients
    target_layer.register_full_backward_hook(lambda module, grad_in, grad_out: save_gradient(grad_out[0]))
    
    # Redo backward to trigger the gradient hook
    model.zero_grad()
    output = model(input_tensor)
    output[:, target_class].backward()
    
    hook.remove()

    # Process Gradients and Feature Maps
    # For ViT, we take the class token or average the spatial tokens
    weights = torch.mean(gradients, dim=1, keepdim=True)
    cam = torch.sum(weights * feature_map, dim=-1)
    
    # Reshape Cam (from patches to 2D)
    # 197 tokens -> remove class token -> 196 -> 14x14 grid
    cam = cam[:, 1:].reshape(1, 14, 14) 
    cam = F.relu(cam)
    cam = cam - cam.min()
    cam = cam / cam.max()
    cam = cam.detach().cpu().numpy()[0]
    
    return cam

# 3. GRADIO PREDICTION FUNCTION
def predict_alzheimer(input_img):
    # Preprocess
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    
    img_pil = Image.fromarray(input_img)
    input_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad = True

    # Get Prediction
    outputs = model(input_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    confidences = {class_names[i]: float(probs[i]) for i in range(4)}
    target_class = torch.argmax(outputs).item()

    # Generate Heatmap (Grad-CAM)
    cam = generate_gradcam(input_tensor, target_class)
    cam = cv2.resize(cam, (224, 224))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Overlay heatmap on original image
    original_res = np.array(img_pil.resize((224, 224)))
    overlay = cv2.addWeighted(original_res, 0.6, heatmap, 0.4, 0)
    
    return confidences, overlay

# 4. LAUNCH GRADIO INTERFACE
interface = gr.Interface(
    fn=predict_alzheimer,
    inputs=gr.Image(label="Upload Brain MRI"),
    outputs=[
        gr.Label(num_top_classes=4, label="Diagnosis Confidence"),
        gr.Image(label="XAI: Grad-CAM Visualization (Where the AI is looking)")
    ],
    title="Alzheimer's Multi-Class Diagnosis (ViT + Grad-CAM)",
    description="Upload a brain MRI scan. The AI will classify the stage and provide a heatmap showing the regions of atrophy detected."
)

interface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a261d93edbf2cc7e32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
